# Day 1: เช็กคุณภาพข้อมูลดิบ

เช็ก `data/raw/openaq_hourly_raw.csv` (510,697 แถว จาก 4 สถานี) ก่อนทำความสะอาดใน `02_cleaning.ipynb`

- 225611 Health Promotion Hospital Maptaput (ใกล้นิคม 4.5 กม.)
- 7846 Field Crop Research Center (ใกล้นิคม 5.8 กม.)
- 717 Pluakdaeng District Health Office (อ้างอิง 33.4 กม.)
- 225690 Agricultural Office (อ้างอิงสำรอง เพิ่มใน Day 4)

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/openaq_hourly_raw.csv')
print(f'จำนวนแถวทั้งหมด: {len(df)}')
print()
print('ค่า min/max/mean แยกตามพารามิเตอร์')
display(df.groupby('parameter')['value'].agg(['min','max','mean']).round(4))
print('จำนวนแถวที่ค่าติดลบ')
neg = df[df['value'] < 0]
print(f'พบ {len(neg)} แถว')


จำนวนแถวทั้งหมด: 510697

ค่า min/max/mean แยกตามพารามิเตอร์


,min,max,mean
parameter,,,
co,0.0,27.100,0.4255
no2,0.0,0.332,0.0085
o3,0.0,0.412,0.0202
pm10,0.0,169.000,42.1884
pm25,1.0,115.000,19.7125
so2,0.0,0.324,0.0027


จำนวนแถวที่ค่าติดลบ
พบ 0 แถว


**สรุป:** ไม่มีค่าติดลบเลย ค่า min/max ของทุกสารอยู่ในช่วงที่สมเหตุสมผล 

## 2. เช็กข้อมูลซ้ำ

นับแถวที่สถานี สาร และเวลาเดียวกันซ้ำกันมากกว่า 1 ครั้ง

In [2]:
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'])
dup = df.groupby(['station_id','parameter','datetime_utc']).size()
dup = dup[dup > 1]
n_groups = df.groupby(['station_id','parameter']).ngroups
print(f'พบ {len(dup)} กลุ่มที่ซ้ำ จากทั้งหมด {n_groups} กลุ่ม station+parameter')


พบ 0 กลุ่มที่ซ้ำ จากทั้งหมด 23 กลุ่ม station+parameter


**สรุป:** ไม่มีข้อมูลซ้ำเลย (0 จาก 23 กลุ่มสถานีและสาร)

## 3. เช็กเซนเซอร์ค้าง

หาช่วงที่ค่าเดิมซ้ำติดกันนานที่สุดของแต่ละสถานีและสาร ถ้านานผิดปกติ เซนเซอร์อาจหยุดทำงานแต่ยังส่งค่าเดิม

In [3]:
df = df.sort_values(['station_id','parameter','datetime_utc'])
results = []
for (sid, param), g in df.groupby(['station_id','parameter']):
    g = g.sort_values('datetime_utc')
    same_as_prev = g['value'].eq(g['value'].shift())
    run_id = (~same_as_prev).cumsum()
    run_lengths = g.groupby(run_id).size()
    results.append({'station_id': sid, 'parameter': param, 'max_stuck_hours': run_lengths.max()})

res_df = pd.DataFrame(results).sort_values('max_stuck_hours', ascending=False)
display(res_df)


,station_id,parameter,max_stuck_hours
0,717,no2,2510
22,225690,so2,179
4,717,so2,128
1,717,o3,127
16,225611,so2,117
12,225611,no2,98
10,7846,so2,52
21,225690,pm25,48
15,225611,pm25,39
20,225690,pm10,39


**พบปัญหา:** NO₂ ของสถานี 717 ค่าเดิมซ้ำติดกันนาน 2,510 ชั่วโมง (ราว 105 วัน) ดูช่วงนี้ต่อ

In [4]:
g = df[(df.station_id==717) & (df.parameter=='no2')].sort_values('datetime_utc').reset_index(drop=True)
same_as_prev = g['value'].eq(g['value'].shift())
run_id = (~same_as_prev).cumsum()
run_lengths = g.groupby(run_id).size()
worst_run = run_lengths.idxmax()
chunk = g[run_id == worst_run]
print('ค่าที่ค้าง:', chunk['value'].iloc[0])
print('ตั้งแต่:', chunk['datetime_utc'].min())
print('ถึง:', chunk['datetime_utc'].max())
print('จำนวนชั่วโมง:', len(chunk))
print()
print('ภาพรวม NO2 ของสถานีนี้ทั้งหมด')
print('รวมทั้งหมด:', len(g), 'ชั่วโมง')
print('ค่าที่พบบ่อยที่สุด 5 อันดับ:')
print(g['value'].value_counts().head())


ค่าที่ค้าง: 0.0
ตั้งแต่: 2022-09-11 03:00:00+00:00
ถึง: 2023-01-30 04:00:00+00:00
จำนวนชั่วโมง: 2510

ภาพรวม NO2 ของสถานีนี้ทั้งหมด
รวมทั้งหมด: 38829 ชั่วโมง
ค่าที่พบบ่อยที่สุด 5 อันดับ:
value
0.000    4132
0.004    3285
0.005    3213
0.003    3146
0.006    2939
Name: count, dtype: int64


**สรุป:** NO₂ ของสถานี 717 ค้างที่ 0.000 ตั้งแต่ 2022-09-11 ถึง 2023-01-30 ซึ่งเป็นวันสุดท้ายที่สถานีนี้ส่งข้อมูล NO₂ อากาศไม่น่าคงที่ที่ 0 นานขนาดนี้ จึงสอดคล้องกับเซนเซอร์เสีย

## 4. ความละเอียดของค่าแต่ละสาร

สารที่มีค่าเป็นไปได้น้อยจะซ้ำกันเองบ่อยกว่า จึงใช้เกณฑ์ "ซ้ำเกิน N ชั่วโมง" เดียวกันทุกสารไม่ได้

In [5]:
for p, g in df.groupby('parameter'):
    v = g['value'].dropna()
    is_int = (v % 1 == 0).all()
    print(f'{p:6s} หน่วย = {g["units"].iloc[0]:7s} ค่าไม่ซ้ำ ={v.nunique():5d}  ช่วง {v.min()}-{v.max()}  เป็นจำนวนเต็มทั้งหมด={is_int}')


co     หน่วย = ppm     ค่าไม่ซ้ำ =  238  ช่วง 0.0-27.1  เป็นจำนวนเต็มทั้งหมด=False
no2    หน่วย = ppm     ค่าไม่ซ้ำ =   87  ช่วง 0.0-0.332  เป็นจำนวนเต็มทั้งหมด=False
o3     หน่วย = ppm     ค่าไม่ซ้ำ =  156  ช่วง 0.0-0.412  เป็นจำนวนเต็มทั้งหมด=False
pm10   หน่วย = µg/m³   ค่าไม่ซ้ำ =  166  ช่วง 0.0-169.0  เป็นจำนวนเต็มทั้งหมด=True
pm25   หน่วย = µg/m³   ค่าไม่ซ้ำ =  886  ช่วง 1.0-115.0  เป็นจำนวนเต็มทั้งหมด=False
so2    หน่วย = ppm     ค่าไม่ซ้ำ =  136  ช่วง 0.0-0.324  เป็นจำนวนเต็มทั้งหมด=False


**สรุป:** NO₂ มีค่าไม่ซ้ำกันแค่ 87 ค่า ส่วน PM2.5 มี 886 ค่า NO₂ จึงซ้ำกันเองง่ายกว่ามาก ต้องแยกเกณฑ์ตามสาร

## 5. ค่า 0 เทียบกับค่าไม่เป็น 0

ค่า 0 ซ้ำสั้น ๆ มักเป็นอากาศสะอาดจนวัดไม่ได้ ส่วนค่าไม่เป็น 0 ที่ซ้ำยาวน่าสงสัยกว่า จึงลองแยกเกณฑ์สองแบบ

In [6]:
rows=[]
for (sid,p), g in df.groupby(['station_id','parameter']):
    g=g.sort_values('datetime_utc')
    run_id=(~g['value'].eq(g['value'].shift())).cumsum()
    runs=g.groupby(run_id).agg(n=('value','size'), val=('value','first'))
    for is_zero, sub in runs.groupby(runs['val']==0):
        rows.append({'station_id':sid,'parameter':p,'zero_run':is_zero,
                     'n_runs':len(sub),'p50':sub['n'].median(),'p99':sub['n'].quantile(.99),'max':sub['n'].max()})
r=pd.DataFrame(rows)
print('ความยาวช่วงค่าซ้ำ แยก ค่า=0 กับ ค่าไม่เป็น 0')
display(r)


ความยาวช่วงค่าซ้ำ แยก ค่า=0 กับ ค่าไม่เป็น 0


,station_id,parameter,zero_run,n_runs,p50,p99,max
0,717,no2,False,28207,1.0,4.00,9
1,717,no2,True,678,2.0,11.23,2510
2,717,o3,False,38632,1.0,3.00,31
3,717,o3,True,1279,3.0,11.00,127
4,717,pm10,False,20072,1.0,10.00,27
5,717,pm10,True,2,12.0,22.78,23
6,717,pm25,False,20402,1.0,10.00,35
7,717,so2,False,9858,1.0,16.00,76
8,717,so2,True,3190,4.0,45.00,128
9,7846,co,False,15459,1.0,3.00,10


**สรุป:** ค่า 0 ซ้ำบ่อยและยาวกว่าค่าไม่เป็น 0 ในแทบทุกสถานีและสาร (เช่น SO₂ ของ 717 มีช่วงค่า 0 ซ้ำ 3,190 ช่วง) จึงติดป้ายว่าน่าสงสัยเมื่อ
- ค่าไม่เป็น 0 ซ้ำติดกัน ≥ 24 ชั่วโมง
- ค่า 0 ซ้ำติดกัน ≥ 72 ชั่วโมง (ผ่อนปรนกว่า เพราะ 0 สั้น ๆ เป็นค่าจริงได้)

## 6. % ข้อมูลที่ถูก flag

วัดผลกระทบของเกณฑ์ก่อนใช้จริงใน `02_cleaning.ipynb`

In [7]:
ZERO_MIN, NONZERO_MIN = 72, 24
out=[]
for (sid,p), g in df.groupby(['station_id','parameter']):
    g=g.sort_values('datetime_utc').copy()
    run_id=(~g['value'].eq(g['value'].shift())).cumsum()
    n=g.groupby(run_id)['value'].transform('size')
    is_zero=g['value'].eq(0)
    flag=((is_zero)&(n>=ZERO_MIN))|((~is_zero)&(n>=NONZERO_MIN))
    out.append({'station_id':sid,'parameter':p,'total_hours':len(g),
                'flagged':int(flag.sum()),'pct':round(100*flag.mean(),2)})
r=pd.DataFrame(out).sort_values('pct',ascending=False)
print(f'% ที่ถูก flag (เกณฑ์: ค่า0 ซ้ำ>={ZERO_MIN}ชม. / ค่าไม่เป็น0 ซ้ำ>={NONZERO_MIN}ชม.)')
display(r)

total_flagged = r.flagged.sum()
total_hours = r.total_hours.sum()
pct = 100 * total_flagged / total_hours
print(f'สรุป: ข้อมูลที่ถูกติดป้ายว่าน่าสงสัย {total_flagged} ชั่วโมง จากทั้งหมด {total_hours} ชั่วโมง คิดเป็น {pct:.2f}%')

% ที่ถูก flag (เกณฑ์: ค่า0 ซ้ำ>=72ชม. / ค่าไม่เป็น0 ซ้ำ>=24ชม.)


,station_id,parameter,total_hours,flagged,pct
22,225690,so2,8051,1991,24.73
0,717,no2,38829,2510,6.46
4,717,so2,47595,1445,3.04
20,225690,pm10,7323,148,2.02
21,225690,pm25,22131,371,1.68
12,225611,no2,6588,98,1.49
16,225611,so2,21352,220,1.03
15,225611,pm25,29517,293,0.99
3,717,pm25,33558,316,0.94
2,717,pm10,44399,201,0.45


สรุป: ข้อมูลที่ถูกติดป้ายว่าน่าสงสัย 7896 ชั่วโมง จากทั้งหมด 510697 ชั่วโมง คิดเป็น 1.55%


## สรุปผลขั้นที่ 1

- ไม่มีค่าติดลบและไม่มีข้อมูลซ้ำ
- เซนเซอร์น่าจะค้างรวม 1.55% (7,896 จาก 510,697 ชั่วโมง)
- PM2.5 และ SO₂ ของ 3 สถานีหลักถูกติดป้ายไม่เกิน 3.04%
- NO₂ ของ 717 ถูกติดป้ายมากสุดในกลุ่มนี้ (6.46%) และเป็นช่วงก่อนสถานีหยุดส่งข้อมูล
- SO₂ ของ 225690 ถูกติดป้ายถึง 24.73% (1,991 จาก 8,051 ชั่วโมง)